<img src="../../../assets/images/logos/ucu_logo_clean.svg" alt="UCU Logo" width="200" style="float: right; margin: 0 0 10px 10px;"/>

### Matemáticas para Aprendizaje Automático - 2026

--------
## Laboratorio 1.2: Proyecciones Ortogonales, Gram-Schmidt y Descomposición QR

#### Objetivos

- Implementar la proyección ortogonal de un vector sobre una recta y verificar las propiedades que la caracterizan.
- Construir la descomposición $QR$ reducida mediante el proceso de Gram-Schmidt y verificar sus propiedades numéricamente.
- Usar la factorización $QR$ para resolver sistemas lineales y para proyectar sobre un subespacio sin invertir matrices.
- Contrastar las implementaciones propias contra `numpy.linalg` y explicar por qué LAPACK no usa Gram-Schmidt clásico.

## Introducción

Buena parte del aprendizaje automático consiste en aproximar un vector de datos por el elemento más cercano dentro de un subespacio: eso es lo que hacen la regresión lineal por mínimos cuadrados, el análisis de componentes principales y la compresión de representaciones. La herramienta que formaliza esa idea es la **proyección ortogonal**: dado $x \in \mathbb{R}^n$ y un subespacio $U \subseteq \mathbb{R}^n$, la proyección $\pi_U(x)$ es el único punto de $U$ que minimiza la distancia $\|x - u\|$ con $u \in U$.

Calcular esa proyección con la fórmula directa $P = B(B^\top B)^{-1}B^\top$ obliga a invertir $B^\top B$, una matriz que suele estar mal condicionada cuando las columnas de $B$ son casi colineales. La alternativa es cambiar de base: si las columnas del subespacio son **ortonormales**, la proyección se reduce a $\hat{Q}\hat{Q}^\top x$ y no hay nada que invertir. A lo largo del laboratorio vas a implementar la **descomposición $QR$** por Gram-Schmidt, usarla para resolver sistemas lineales y para proyectar, y compararla contra `numpy.linalg` en casos bien y mal condicionados. Cada implementación se valida contra un resultado analítico conocido o contra la librería de referencia.

In [ ]:
# Librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
import timeit


def coseno(u, v):
    """
    Calcula el coseno del ángulo entre dos vectores no nulos:

        cos(u, v) = <u, v> / (||u|| ||v||)

    Esta función viene dada. Se usa en todo el laboratorio para verificar
    ortogonalidad (coseno ~ 0) y colinealidad (coseno ~ ±1).
    """
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

## Parte 1: Proyecciones Ortogonales

El **producto interno** estándar de $\mathbb{R}^n$ es $\langle x, y \rangle = x^\top y = \sum_{i=1}^n x_i y_i$, y la norma asociada es $\|x\| = \sqrt{x^\top x}$. Dos vectores son **ortogonales** cuando $x^\top y = 0$. El ángulo $\theta$ entre dos vectores no nulos queda determinado por

$$\cos\theta = \frac{x^\top y}{\|x\|\,\|y\|} \in [-1, 1],$$

cociente que vale $0$ si los vectores son ortogonales y $\pm 1$ si son colineales. La función `coseno` de la celda anterior lo calcula, y se usa como test numérico de ambas propiedades.

**Proyección sobre una recta.** Sea $b \in \mathbb{R}^n$ no nulo y $U = \operatorname{span}\{b\}$. Buscamos $\pi_U(x) = \lambda b$ tal que el residuo $x - \lambda b$ sea ortogonal a $b$. Imponiendo $b^\top (x - \lambda b) = 0$ se despeja

$$\lambda = \frac{b^\top x}{b^\top b}, \qquad \pi_U(x) = \frac{b\,b^\top}{b^\top b}\,x.$$

La matriz $P_\pi = \dfrac{b\,b^\top}{b^\top b}$ realiza la proyección para cualquier $x$. Cumple dos propiedades que la caracterizan: es **idempotente** ($P^2 = P$, proyectar dos veces es igual que proyectar una) y **simétrica** ($P^\top = P$).

**Proyección sobre un subespacio general.** Sea $B \in \mathbb{R}^{n \times m}$ con columnas linealmente independientes y $U = \operatorname{col}(B)$. Buscamos $\pi_U(x) = B\lambda$ con $\lambda \in \mathbb{R}^m$ tal que el residuo sea ortogonal a **todas** las columnas de $B$:

$$B^\top (x - B\lambda) = 0 \;\Longleftrightarrow\; B^\top B\,\lambda = B^\top x.$$

Esta es la **ecuación normal**. Como las columnas son linealmente independientes, $B^\top B \in \mathbb{R}^{m \times m}$ es invertible y

$$\lambda = (B^\top B)^{-1} B^\top x, \qquad \pi_U(x) = B\lambda, \qquad P_\pi = B(B^\top B)^{-1}B^\top.$$

La proyección sobre una recta es el caso $m = 1$ de esta fórmula. En esta parte vas a implementar el caso de la recta y verificarlo con dos tests: ortogonalidad del residuo e idempotencia de $P_\pi$. El caso general sobre un subespacio lo vas a resolver en la Parte 3, donde la base ortonormal que da la $QR$ vuelve innecesaria la ecuación normal.

### Ejercicio L1.2.1: Proyección sobre una Recta

Implementá las dos formas de proyectar sobre la recta $\operatorname{span}\{b\}$ presentadas en la introducción.

**a)** `proyeccion_recta(b, x)`: devuelve la tupla `(proj, lam)` con la proyección $\pi = \lambda b$ y el coeficiente $\lambda = \dfrac{b^\top x}{b^\top b}$.

**b)** `matriz_proyeccion_recta(b)`: devuelve la matriz $P_\pi = \dfrac{b\,b^\top}{b^\top b} \in \mathbb{R}^{n \times n}$.

**Nota:** para construir $b\,b^\top$ a partir de un arreglo 1D usá `np.outer(b, b)`. El producto interno se calcula con `np.dot`.

In [ ]:
def proyeccion_recta(b, x):
    """
    Calcula la proyección ortogonal de x sobre la recta generada por b.

    Args:
        b: vector que define la recta (array 1D de largo n)
        x: vector a proyectar (array 1D de largo n)

    Returns:
        proj: la proyección de x sobre span{b} (array 1D de largo n)
        lam: el coeficiente (coordenada) de la proyección (escalar)
    """
    lam = ...   # COMPLETAR
    proj = ...  # COMPLETAR
    return proj, lam


def matriz_proyeccion_recta(b):
    """
    Calcula la matriz de proyección P sobre la recta generada por b.

    Args:
        b: vector que define la recta (array 1D de largo n)

    Returns:
        P: matriz de proyección (n x n)
    """
    P = ...  # COMPLETAR
    return P


# ── Verificación ─────────────────────────────────────────────────────────────
# Caso analítico: b = [1, -1, 2], x = [3, 1, 0].
#   lambda = (b^T x) / (b^T b) = 2 / 6 = 1/3   =>   pi = [1/3, -1/3, 2/3].
b_test = np.array([1.0, -1.0, 2.0])
x_test = np.array([3.0, 1.0, 0.0])

proj, lam = proyeccion_recta(b_test, x_test)
P = matriz_proyeccion_recta(b_test)

esperado_proj = np.array([1 / 3, -1 / 3, 2 / 3])

assert np.allclose(proj, esperado_proj), "La proyección no coincide con el valor analítico"
assert np.isclose(lam, 1 / 3), "El coeficiente lambda no coincide con el analítico"
assert np.allclose(P @ x_test, proj), "Ambos métodos deben dar la misma proyección"
print("Obtenido : pi =", proj, "| lambda =", lam)
print("Esperado : pi =", esperado_proj, "| lambda =", 1 / 3)

### Ejercicio L1.2.2: Propiedades de la Proyección

Con $b = [1, -1, 2]^\top$ y $x = [3, 1, 0]^\top$ del ejercicio anterior, comprobá las tres propiedades que caracterizan a una proyección ortogonal.

**a)** Calculá el residuo $r = x - \pi$ y verificá con `coseno` que es ortogonal a $b$.

**b)** Verificá que $P_\pi$ es idempotente, es decir $P^2 = P$.

El bloque de verificación agrega por su cuenta los dos controles restantes: que $\pi$ es colineal con $b$ y que $P_\pi$ es simétrica.

**Nota:** el producto matricial en NumPy es `@`. Cuidado con `*`, que multiplica elemento a elemento.

In [ ]:
b = np.array([1.0, -1.0, 2.0])
x = np.array([3.0, 1.0, 0.0])

pi, lam = proyeccion_recta(b, x)
P = matriz_proyeccion_recta(b)

# a) Residuo y ortogonalidad
r = ...        # COMPLETAR: residuo x - pi
cos_res = ...  # COMPLETAR: coseno entre el residuo y b

# b) Idempotencia
P2 = ...       # COMPLETAR: P al cuadrado (producto matricial)

# ── Verificación ─────────────────────────────────────────────────────────────
esperado_r = np.array([8 / 3, 4 / 3, -2 / 3])

assert np.allclose(r, esperado_r), "El residuo no coincide con el valor analítico"
assert np.isclose(cos_res, 0.0, atol=1e-12), "El residuo debe ser ortogonal a b"
assert np.allclose(P2, P), "P debe ser idempotente"
assert np.isclose(abs(coseno(pi, b)), 1.0), "La proyección debe ser colineal con b"
assert np.allclose(P.T, P), "P debe ser simétrica"
print("Obtenido : r =", r, "| cos(r, b) =", cos_res)
print("Esperado : r =", esperado_r, "| cos(r, b) =", 0.0)
print("cos(pi, b) =", coseno(pi, b), "| P simétrica:", np.allclose(P.T, P))

## Parte 2: Gram-Schmidt y Descomposición QR

Trabajar con una base ortonormal simplifica las cuentas: si $\{q_1, \dots, q_k\}$ es ortonormal, las coordenadas de un vector en esa base son productos internos y no hay ningún sistema que resolver. El proceso de **Gram-Schmidt** construye esa base a partir de una base cualquiera, procesando los vectores en orden.

La idea es la de la Parte 1: para ortogonalizar $v_j$ contra los $q_i$ ya construidos, se le resta su proyección sobre cada uno de ellos y se normaliza el resultado. Como los $q_i$ son ortonormales, la proyección sobre $q_i$ se reduce a $(q_i^\top v_j)\,q_i$, sin denominador:

$$\tilde{v}_j = v_j - \sum_{i=1}^{j-1} (q_i^\top v_j)\,q_i, \qquad q_j = \frac{\tilde{v}_j}{\|\tilde{v}_j\|}, \qquad j = 1, \dots, k.$$

Por construcción $q_j$ es ortogonal a todos los $q_i$ con $i < j$, tiene norma $1$, y en cada paso se preserva el subespacio generado: $\operatorname{span}\{q_1, \dots, q_j\} = \operatorname{span}\{v_1, \dots, v_j\}$. Si los vectores de entrada son **linealmente dependientes**, en algún paso $\tilde{v}_j = 0$ y la normalización queda indefinida; esa situación se detecta comparando $\|\tilde{v}_j\|$ contra una tolerancia.

**De Gram-Schmidt a la QR.** El proceso descarta información: los coeficientes $q_i^\top v_j$ que se restan y las normas con las que se divide. Despejando $v_j$ de la recurrencia, cada columna original se escribe como combinación lineal de las $q_i$ con $i \le j$:

$$v_j = \sum_{i<j} (q_i^\top v_j)\,q_i + \|\tilde{v}_j\|\,q_j.$$

Guardando $R_{ij} = q_i^\top v_j$ para $i < j$ y $R_{jj} = \|\tilde{v}_j\|$, esa igualdad para todas las columnas es exactamente $A = \hat{Q}\hat{R}$. Para $A \in \mathbb{R}^{m \times n}$ con $m \ge n$ y columnas linealmente independientes, la **descomposición $QR$ reducida** es

$$A = \hat{Q}\hat{R}, \qquad \hat{Q} \in \mathbb{R}^{m \times n} \text{ con } \hat{Q}^\top \hat{Q} = I_n, \qquad \hat{R} \in \mathbb{R}^{n \times n} \text{ triangular superior}.$$

Como $R_{jj} = \|\tilde{v}_j\| > 0$, la diagonal que produce Gram-Schmidt es positiva; con esa convención la factorización es única.

**Qué hace NumPy.** `np.linalg.qr(A, mode=...)` admite varios modos: `'reduced'` (el de arriba, por defecto), `'complete'` (devuelve $Q \in \mathbb{R}^{m \times m}$ y $R \in \mathbb{R}^{m \times n}$) y `'raw'`, que expone la salida cruda de LAPACK. Por debajo no usa Gram-Schmidt sino **reflexiones de Householder**, que nunca construyen $Q$ explícitamente durante el proceso y son más estables. Una consecuencia visible es que los signos de las columnas pueden diferir de los tuyos: si $S$ es diagonal con $\pm 1$, entonces $(\hat{Q}S)(S\hat{R}) = \hat{Q}\hat{R} = A$, así que ambas son factorizaciones válidas. En el Ejercicio L1.2.5 vas a medir cuánto se separan las dos implementaciones cuando la matriz está mal condicionada.

### Ejercicio L1.2.3: Descomposición QR vía Gram-Schmidt

Implementá `qr_gram_schmidt(A)`, que calcula la $QR$ reducida de $A \in \mathbb{R}^{m \times n}$ con $m \ge n$ aplicando Gram-Schmidt a las columnas y guardando los coeficientes en $R$. Devuelve la tupla `(Q, R)`.

El andamiaje trae la estructura de bucles y el control de dependencia lineal. Completá las tres asignaciones: el coeficiente fuera de la diagonal $R_{ij} = q_i^\top v_j$, el elemento diagonal $R_{jj} = \|\tilde{v}_j\|$, y la normalización $q_j = \tilde{v}_j / R_{jj}$.

**Nota:** el andamiaje resta la proyección inmediatamente después de calcular cada $R_{ij}$, lo que en aritmética exacta es equivalente a la fórmula de la introducción.

In [ ]:
def qr_gram_schmidt(A):
    """
    Calcula la descomposición QR reducida de A usando Gram-Schmidt.

    Args:
        A: matriz de tamaño m x n con columnas linealmente independientes (m >= n)

    Returns:
        Q: matriz con columnas ortonormales (m x n)
        R: matriz triangular superior con diagonal positiva (n x n)
    """
    m, n = A.shape
    Q = np.zeros((m, n), dtype=float)
    R = np.zeros((n, n), dtype=float)

    for j in range(n):
        v = A[:, j].astype(float).copy()

        for i in range(j):
            R[i, j] = ...  # COMPLETAR: coeficiente q_i^T v
            v = v - R[i, j] * Q[:, i]

        R[j, j] = ...  # COMPLETAR: norma del vector ortogonalizado

        if R[j, j] < 1e-10:
            raise ValueError(f"La columna {j} es linealmente dependiente de las anteriores")

        Q[:, j] = ...  # COMPLETAR: normalizar v

    return Q, R


# ── Verificación ─────────────────────────────────────────────────────────────
# Caso analítico: A = [[3, 1], [0, 2]] ya tiene columnas ortogonales en el orden
# correcto, así que Q = I y R = A.
A_test = np.array([[3.0, 1.0],
                   [0.0, 2.0]])
Q_test, R_test = qr_gram_schmidt(A_test)

assert np.allclose(Q_test, np.eye(2)), "Q no coincide con el esperado"
assert np.allclose(R_test, A_test), "R no coincide con el esperado"
assert np.allclose(Q_test @ R_test, A_test), "QR debe reconstruir A"
print("Obtenido : Q =\n", Q_test, "\nR =\n", R_test)
print("Esperado : Q =\n", np.eye(2), "\nR =\n", A_test)

### Ejercicio L1.2.4: Verificación de la Factorización

Aplicá `qr_gram_schmidt` a

$$A = \begin{bmatrix} 1 & 1 & 0 \\ 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix}$$

y comprobá las propiedades que definen la factorización.

**a)** Las columnas de $Q$ son ortogonales entre sí: calculá el coseno de cada par de columnas distintas.

**b)** $R$ es triangular superior: extraé su parte estrictamente inferior con `np.tril(R, k=-1)`.

**c)** Gram-Schmidt preserva el subespacio generado. Como $\operatorname{col}(Q) = \operatorname{col}(V)$ y $Q$ tiene columnas ortonormales, proyectar las columnas de $V$ sobre $\operatorname{col}(Q)$ las deja fijas: verificalo calculando $QQ^\top V$ para la matriz rectangular $V \in \mathbb{R}^{4 \times 2}$ del andamiaje.

**d)** El algoritmo detecta la dependencia lineal. Aplicalo a $b_1 = [1,2,0]^\top$ y $b_2 = [3,6,0]^\top$ dentro de un bloque `try/except` y guardá el mensaje de error. Como $b_2 = 3b_1$, al restarle su proyección sobre $q_1$ queda el vector nulo.

**Nota:** para recorrer los pares de columnas distintas conviene una comprensión de lista con dos índices, `for i in range(n) for j in range(i + 1, n)`.

In [ ]:
A = np.array([[1.0, 1.0, 0.0],
              [1.0, 0.0, 1.0],
              [0.0, 1.0, 1.0]])
Q, R = qr_gram_schmidt(A)

# a) Ortogonalidad de las columnas de Q
cos_pares = ...     # COMPLETAR: array con el coseno de cada par de columnas distintas de Q

# b) R triangular superior
tri_inferior = ...  # COMPLETAR: parte estrictamente inferior de R

# c) Gram-Schmidt preserva el subespacio generado
V = np.array([[1.0, 1.0],
              [1.0, 0.0],
              [0.0, 1.0],
              [0.0, 0.0]])
Qv, Rv = qr_gram_schmidt(V)
V_reproy = ...      # COMPLETAR: Qv Qv^T V

# d) Detección de columnas linealmente dependientes
B_ld = np.array([[1.0, 3.0],
                 [2.0, 6.0],
                 [0.0, 0.0]])
error_ld = None
try:
    ...  # COMPLETAR: llamar a qr_gram_schmidt(B_ld)
except ValueError as e:
    error_ld = str(e)

# ── Verificación ─────────────────────────────────────────────────────────────
esperado_diag = np.array([np.sqrt(2), np.sqrt(3 / 2), 2 / np.sqrt(3)])

assert np.allclose(Q @ R, A), "QR debe reconstruir A"
assert np.allclose(cos_pares, 0.0, atol=1e-12), "Las columnas de Q deben ser ortogonales"
assert np.allclose(np.linalg.norm(Q, axis=0), 1.0), "Las columnas de Q deben tener norma 1"
assert np.allclose(tri_inferior, 0.0), "R debe ser triangular superior"
assert np.all(np.diag(R) > 0), "La diagonal de R debe ser positiva"
assert np.allclose(np.diag(R), esperado_diag), "La diagonal de R no coincide con la analítica"
assert np.allclose(V_reproy, V), "Q debe generar el mismo subespacio que V"
assert error_ld is not None, "El algoritmo debe fallar con columnas linealmente dependientes"
print("Obtenido : diag(R) =", np.diag(R))
print("Esperado : diag(R) =", esperado_diag)
print("cos entre pares de columnas de Q :", cos_pares)
print("Error con columnas l.d. :", error_ld)

### Ejercicio L1.2.5: Comparación contra `numpy.linalg.qr`

Compará tu factorización contra la de NumPy usando dos medidas: el **error de reconstrucción** $\|QR - A\|_F$ y el **error de ortogonalidad** $\|Q^\top Q - I\|_F$. La función `errores_qr` que las calcula viene dada.

**a)** Factorizá con `np.linalg.qr` en modo reducido la matriz $A$ del ejercicio anterior.

**b)** Factorizá con tu implementación una matriz aleatoria de $50 \times 10$. El andamiaje ya calcula la versión de NumPy y arma la tabla comparativa.

**c)** Factorizá con tu implementación la **matriz de Hilbert** de $8 \times 8$, definida por $H_{ij} = 1/(i+j+1)$, cuyo número de condición es del orden de $10^{10}$. Compará su error de ortogonalidad contra el de NumPy y observá la diferencia: son varios órdenes de magnitud. Esa es la razón por la que LAPACK implementa la $QR$ con reflexiones de Householder y no con Gram-Schmidt.

**Nota:** los signos de las columnas pueden diferir entre implementaciones, así que la comparación se hace sobre los errores y no elemento a elemento. El modo reducido se pide con `np.linalg.qr(M, mode="reduced")`.

In [ ]:
np.random.seed(0)


def errores_qr(M, Qm, Rm):
    """Devuelve (error de reconstrucción, error de ortogonalidad) en norma de Frobenius."""
    err_rec = np.linalg.norm(Qm @ Rm - M)
    err_ort = np.linalg.norm(Qm.T @ Qm - np.eye(Qm.shape[1]))
    return err_rec, err_ort


# a) La matriz A del ejercicio anterior, con NumPy
Q_np, R_np = ...    # COMPLETAR: QR reducida de A con numpy

# b) Matriz aleatoria de 50 x 10
A_rand = np.random.randn(50, 10)
Q_r, R_r = ...      # COMPLETAR: QR de A_rand con tu implementación
Q_rnp, R_rnp = np.linalg.qr(A_rand, mode="reduced")

# c) Matriz de Hilbert de 8 x 8 (mal condicionada)
n_h = 8
H = 1.0 / (np.arange(n_h)[:, None] + np.arange(n_h)[None, :] + 1.0)
Q_h, R_h = ...      # COMPLETAR: QR de H con tu implementación
Q_hnp, R_hnp = np.linalg.qr(H, mode="reduced")

print(f"{'caso':<22}{'||QR - A||_F':>16}{'||Q^T Q - I||_F':>18}")
for nombre, M, Qm, Rm in [("A (3x3) propia", A, Q, R),
                          ("A (3x3) numpy", A, Q_np, R_np),
                          ("aleatoria 50x10 propia", A_rand, Q_r, R_r),
                          ("aleatoria 50x10 numpy", A_rand, Q_rnp, R_rnp),
                          ("Hilbert 8x8 propia", H, Q_h, R_h),
                          ("Hilbert 8x8 numpy", H, Q_hnp, R_hnp)]:
    err_rec, err_ort = errores_qr(M, Qm, Rm)
    print(f"{nombre:<22}{err_rec:>16.3e}{err_ort:>18.3e}")
print(f"\nnúmero de condición de H : {np.linalg.cond(H):.3e}")

# ── Verificación ─────────────────────────────────────────────────────────────
assert np.allclose(Q_np @ R_np, A), "La factorización de numpy debe reconstruir A"
assert np.allclose(Q_r @ R_r, A_rand), "Tu factorización debe reconstruir A_rand"
assert np.allclose(Q_h @ R_h, H), "Tu factorización debe reconstruir H"
assert errores_qr(A_rand, Q_r, R_r)[1] < 1e-8, "Con una matriz bien condicionada Q debe ser ortonormal"
assert errores_qr(H, Q_h, R_h)[1] > errores_qr(H, Q_hnp, R_hnp)[1], \
    "Con la matriz de Hilbert, Gram-Schmidt debe perder más ortogonalidad que numpy"
print("\nObtenido : ortogonalidad Hilbert propia =", f"{errores_qr(H, Q_h, R_h)[1]:.3e}")
print("Esperado : varios órdenes de magnitud peor que numpy =",
      f"{errores_qr(H, Q_hnp, R_hnp)[1]:.3e}")

## Parte 3: Aplicaciones de la Descomposición QR

Con la factorización disponible, dos problemas de la Parte 1 se resuelven sin invertir nada.

**Resolución de sistemas lineales.** Si $A$ es cuadrada e invertible, $Ax = b$ equivale a $\hat{Q}\hat{R}x = b$. Multiplicando por $\hat{Q}^\top$ y usando $\hat{Q}^\top\hat{Q} = I$:

$$\hat{R}x = \hat{Q}^\top b.$$

Queda un sistema triangular superior, que se resuelve por **sustitución hacia atrás** desde la última incógnita hacia la primera:

$$x_i = \frac{1}{R_{ii}}\left(y_i - \sum_{j=i+1}^{n} R_{ij}\,x_j\right), \qquad i = n, n-1, \dots, 1.$$

El costo de esta etapa es $O(n^2)$, frente a los $O(n^3)$ de calcular la factorización.

**Proyección ortogonal.** Sea $B = \hat{Q}\hat{R}$ la $QR$ reducida de $B \in \mathbb{R}^{n \times m}$, con $\hat{R}$ invertible por tener diagonal positiva. Sustituyendo en la matriz de proyección de la Parte 1:

$$P_\pi = B(B^\top B)^{-1}B^\top = \hat{Q}\hat{R}\left(\hat{R}^\top \hat{Q}^\top \hat{Q}\hat{R}\right)^{-1}\hat{R}^\top \hat{Q}^\top.$$

Como $\hat{Q}^\top\hat{Q} = I_m$, el paréntesis interior es $\hat{R}^\top\hat{R}$, y usando $(\hat{R}^\top\hat{R})^{-1} = \hat{R}^{-1}\hat{R}^{-\top}$ todo se cancela:

$$P_\pi = \hat{Q}\hat{R}\,\hat{R}^{-1}\hat{R}^{-\top}\hat{R}^\top \hat{Q}^\top = \hat{Q}\hat{Q}^\top.$$

El resultado tiene una lectura directa: $\hat{Q}^\top x$ son las coordenadas de $x$ en la base ortonormal $\{q_1, \dots, q_m\}$, y $\hat{Q}$ las vuelve a expresar en $\mathbb{R}^n$.

La diferencia práctica entre las dos rutas está en el condicionamiento. La ecuación normal resuelve un sistema con $B^\top B$, cuyo número de condición es el **cuadrado** del de $B$: si las columnas son casi colineales, la pérdida de precisión se duplica en dígitos. La versión $QR$ nunca forma $B^\top B$. Como contrapartida, calcular la factorización cuesta $O(nm^2)$, así que para proyectar un único vector sobre un subespacio bien condicionado la ecuación normal puede resultar más rápida.

### Ejercicio L1.2.6: Resolución de Sistemas Lineales

**a)** Implementá `sustitucion_atras(R, y)`, que resuelve $Rx = y$ con $R$ triangular superior aplicando la recurrencia de la introducción, recorriendo $i$ desde $n-1$ hasta $0$ (índices de Python).

**b)** Implementá `resolver_qr(A, b)`, que factoriza $A = QR$ con `qr_gram_schmidt`, calcula $y = Q^\top b$ y devuelve la solución de $Rx = y$ obtenida con `sustitucion_atras`.

**Nota:** la suma del paso $i$ se escribe de forma vectorizada como `R[i, i+1:] @ x[i+1:]`. Cuando $i$ es el último índice, ambos cortes quedan vacíos y el producto da $0$, así que no hace falta un caso especial.

In [ ]:
def sustitucion_atras(R, y):
    """
    Resuelve R x = y por sustitución hacia atrás, con R triangular superior.

    Args:
        R: matriz triangular superior con diagonal no nula (n x n)
        y: vector del lado derecho (array 1D de largo n)

    Returns:
        x: solución del sistema (array 1D de largo n)
    """
    n = R.shape[0]
    x = np.zeros(n, dtype=float)
    for i in range(n - 1, -1, -1):
        x[i] = ...  # COMPLETAR: despejar x[i] usando las componentes ya calculadas
    return x


def resolver_qr(A, b):
    """
    Resuelve A x = b usando la descomposición QR reducida.

    Args:
        A: matriz cuadrada invertible (n x n)
        b: vector del lado derecho (array 1D de largo n)

    Returns:
        x: solución del sistema (array 1D de largo n)
    """
    Q, R = ...  # COMPLETAR: factorizar A
    y = ...     # COMPLETAR: Q^T b
    x = ...     # COMPLETAR: resolver R x = y
    return x


# ── Verificación ─────────────────────────────────────────────────────────────
# Caso analítico: R = [[2, 1], [0, 3]], y = [4, 6].
#   x_2 = 6/3 = 2,  x_1 = (4 - 1*2)/2 = 1   =>   x = [1, 2].
R_test = np.array([[2.0, 1.0],
                   [0.0, 3.0]])
y_test = np.array([4.0, 6.0])

obtenido = sustitucion_atras(R_test, y_test)
esperado = np.array([1.0, 2.0])

assert np.allclose(obtenido, esperado), "El resultado no coincide con el esperado"
assert np.allclose(R_test @ obtenido, y_test), "La solución debe satisfacer R x = y"
print("Obtenido :", obtenido)
print("Esperado :", esperado)

### Ejercicio L1.2.7: Validación y Costo Computacional

**a)** Resolvé $Ax = b$ con

$$A = \begin{bmatrix} 2 & 1 & 1 \\ 4 & 3 & 3 \\ 8 & 7 & 9 \end{bmatrix}, \qquad b = [5, 13, 33]^\top,$$

usando `resolver_qr` y usando `numpy.linalg.solve`. La solución analítica es $x = [1, 1, 2]^\top$.

**b)** Resolvé con `resolver_qr` un sistema aleatorio de $20 \times 20$. El andamiaje verifica que coincide con NumPy, mide los tiempos de ambos con `timeit` y grafica el costo de `resolver_qr` para $n \in \{10, 20, 50, 100\}$ contra una referencia $n^3$.

Al ejecutar, comparé la pendiente de las dos curvas del gráfico y el cociente entre los dos tiempos. La diferencia de velocidad no viene del algoritmo (NumPy también factoriza) sino de que LAPACK está compilado y opera por bloques sobre la memoria caché, mientras que el bucle de Python interpreta cada iteración.

In [ ]:
np.random.seed(7)

# a) Sistema 3 x 3
A_sis = np.array([[2.0, 1.0, 1.0],
                  [4.0, 3.0, 3.0],
                  [8.0, 7.0, 9.0]])
b_sis = np.array([5.0, 13.0, 33.0])

x_qr = ...  # COMPLETAR: resolver con resolver_qr
x_np = ...  # COMPLETAR: resolver con numpy.linalg.solve

# b) Sistema aleatorio 20 x 20
n = 20
A_rand20 = np.random.randn(n, n)
b_rand20 = np.random.randn(n)

x_qr_rand = ...  # COMPLETAR: resolver A_rand20 x = b_rand20 con resolver_qr
x_np_rand = np.linalg.solve(A_rand20, b_rand20)

t_qr = timeit.timeit(lambda: resolver_qr(A_rand20, b_rand20), number=100)
t_np = timeit.timeit(lambda: np.linalg.solve(A_rand20, b_rand20), number=100)
print(f"resolver_qr        : {t_qr:.4f} s (100 ejecuciones)")
print(f"numpy.linalg.solve : {t_np:.4f} s (100 ejecuciones)")
print(f"cociente           : {t_qr / t_np:.1f}x")

tamanios = [10, 20, 50, 100]
tiempos = []
for k in tamanios:
    Ak = np.random.randn(k, k)
    bk = np.random.randn(k)
    tiempos.append(timeit.timeit(lambda: resolver_qr(Ak, bk), number=10) / 10)

plt.figure(figsize=(5, 3.5))
plt.loglog(tamanios, tiempos, "o-", label="resolver_qr")
plt.loglog(tamanios, [tiempos[0] * (k / tamanios[0]) ** 3 for k in tamanios], "--", label="$n^3$")
plt.xlabel("n"); plt.ylabel("tiempo por resolución [s]")
plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()

# ── Verificación ─────────────────────────────────────────────────────────────
esperado = np.array([1.0, 1.0, 2.0])

assert np.allclose(x_qr, esperado), "La solución no coincide con la analítica"
assert np.allclose(x_qr, x_np), "resolver_qr y numpy.linalg.solve deben coincidir"
assert np.allclose(A_sis @ x_qr, b_sis), "La solución debe satisfacer A x = b"
assert np.allclose(x_qr_rand, x_np_rand, atol=1e-8), "Ambos métodos deben coincidir"
assert np.allclose(A_rand20 @ x_qr_rand, b_rand20, atol=1e-8), "La solución debe satisfacer A x = b"
print("Obtenido :", x_qr)
print("Esperado :", esperado)

### Ejercicio L1.2.8: Proyección vía QR

**a)** Implementá `proyeccion_via_qr(B, x)`, que factoriza $B = \hat{Q}\hat{R}$ con `qr_gram_schmidt` y devuelve $\pi = \hat{Q}\hat{Q}^\top x$.

**b)** Aplicala a $x = [4, 4, 0]^\top$ y $U = \operatorname{span}\{[1,0,1]^\top,\ [0,1,1]^\top\}$, y verificá con `coseno` que el residuo es ortogonal a cada columna de $B$. El resultado analítico es $\pi = \left[\tfrac{4}{3}, \tfrac{4}{3}, \tfrac{8}{3}\right]^\top$.

El andamiaje agrega dos comprobaciones ya escritas: que $\hat{Q}\hat{Q}^\top$ coincide con la matriz de proyección $B(B^\top B)^{-1}B^\top$ de la Parte 1, y el comportamiento del método sobre un subespacio mal condicionado, $B = [v,\ v + \varepsilon w]$ con $\varepsilon = 10^{-8}$, donde $B^\top B$ sería casi singular y la ruta $QR$ igual mantiene el residuo ortogonal.

**Nota:** no hace falta construir $\hat{Q}\hat{Q}^\top$ explícitamente para proyectar. Calcular `Q @ (Q.T @ x)` evita formar una matriz $n \times n$.

In [ ]:
np.random.seed(3)


def proyeccion_via_qr(B, x):
    """
    Calcula la proyección ortogonal de x sobre col(B) usando la QR reducida.

    Args:
        B: matriz cuyas columnas generan el subespacio (n x m)
        x: vector a proyectar (array 1D de largo n)

    Returns:
        proj: la proyección de x sobre col(B) (array 1D de largo n)
    """
    Q, R = ...  # COMPLETAR: factorizar B
    proj = ...  # COMPLETAR: aplicar Q Q^T a x
    return proj


# b) Aplicación al caso concreto
B_p = np.array([[1.0, 0.0],
                [0.0, 1.0],
                [1.0, 1.0]])
x_p = np.array([4.0, 4.0, 0.0])

pi_qr = ...   # COMPLETAR: proyección vía QR
cos_qr = ...  # COMPLETAR: coseno entre el residuo x_p - pi_qr y cada columna de B_p

# Q Q^T coincide con la matriz de proyección de la Parte 1
Q_p, _ = qr_gram_schmidt(B_p)
P_qr = Q_p @ Q_p.T
P_normal = B_p @ np.linalg.inv(B_p.T @ B_p) @ B_p.T

# Subespacio mal condicionado: B = [v, v + eps w]
eps = 1e-8
v = np.random.randn(50)
w = np.random.randn(50)
B_mal = np.column_stack([v, v + eps * w])
x_mal = np.random.randn(50)
pi_mal_qr = proyeccion_via_qr(B_mal, x_mal)
err_qr = max(abs(coseno(x_mal - pi_mal_qr, B_mal[:, j])) for j in range(2))
print(f"número de condición de B_mal : {np.linalg.cond(B_mal):.3e}")
print(f"mal condicionado, vía QR     : max |cos(r, B_j)| = {err_qr:.3e}")

# ── Verificación ─────────────────────────────────────────────────────────────
esperado_pi = np.array([4 / 3, 4 / 3, 8 / 3])

assert np.allclose(pi_qr, esperado_pi), "La proyección vía QR no coincide con la analítica"
assert np.allclose(cos_qr, 0.0, atol=1e-12), "El residuo debe ser ortogonal a col(B)"
assert np.allclose(P_qr, P_normal), "Q Q^T debe coincidir con B (B^T B)^-1 B^T"
print("\nObtenido : pi_qr =", pi_qr)
print("Esperado :", esperado_pi)

## Conclusiones

En este laboratorio hemos explorado:

1. **Proyecciones ortogonales**: la proyección se obtiene imponiendo que el residuo sea ortogonal al subespacio. Sobre una recta eso da $\lambda = b^\top x / b^\top b$; sobre un subespacio general conduce a la ecuación normal $B^\top B \lambda = B^\top x$. La matriz de proyección resultante es idempotente y simétrica, y la proyección es el punto del subespacio más cercano al vector original.
2. **Gram-Schmidt y descomposición QR**: restar a cada vector su proyección sobre los anteriores y normalizar produce una base ortonormal del mismo subespacio; guardar los coeficientes que ese proceso descarta convierte el algoritmo en la factorización $A = \hat{Q}\hat{R}$, con $\hat{R}$ triangular superior de diagonal positiva.
3. **Aplicaciones de la QR**: resolver $Ax = b$ se reduce a $y = \hat{Q}^\top b$ seguido de una sustitución hacia atrás de costo $O(n^2)$, y la matriz de proyección se simplifica a $\hat{Q}\hat{Q}^\top$, sin invertir nada.
4. **Precisión y costo frente a NumPy**: en matrices bien condicionadas las implementaciones propias coinciden con `numpy.linalg` hasta el error de máquina. Sobre la matriz de Hilbert, en cambio, Gram-Schmidt pierde varios órdenes de magnitud de ortogonalidad, que es la razón por la que LAPACK usa reflexiones de Householder. La misma tensión aparece entre la ecuación normal, que opera con $B^\top B$ y eleva al cuadrado el número de condición, y la ruta $QR$, que nunca forma ese producto y es más estable, aunque cuesta $O(nm^2)$. Esa compensación entre estabilidad numérica y tiempo de cómputo reaparece en mínimos cuadrados, regresión lineal y PCA.

## Declaración de uso de inteligencia artificial

> **Política del curso (syllabus).** Se permite usar herramientas de IA (ChatGPT, Copilot, Claude, etc.)
> *como apoyo para el aprendizaje*: entender conceptos, explorar ideas, depurar código o buscar
> explicaciones alternativas. **No** se permite usarlas para **resolver los ejercicios evaluados** ni
> para **verificar las respuestas antes de entregar**. Se espera que cada estudiante resuelva todos los
> problemas por sí mismo/a.

Completá esta declaración **escribiendo tu respuesta** donde aparece «…» (doble clic en esta celda para
editarla y luego Ctrl/Cmd + Enter para volver a renderizarla):

**1. ¿Usaste herramientas de IA en este laboratorio?** (Sí / No): «…»

**2. ¿Cuál(es)?** (ChatGPT, Copilot, Claude, …; escribí "ninguna" si no usaste): «…»

**3. ¿Para qué la(s) usaste?** Usos permitidos: entender conceptos, explorar ideas, depurar código,
buscar explicaciones alternativas. Escribí los que apliquen: «…»

**4. Detalle breve:** «En qué ejercicios y de qué manera. Ej.: "Usé Claude para entender el proceso de
Gram-Schmidt antes del ejercicio L1.2.3."»

**Declaración de honestidad académica.** Declaro que resolví los ejercicios de este laboratorio por mí
mismo/a y que no utilicé herramientas de IA para resolver los ejercicios evaluados ni para verificar mis
respuestas antes de entregar, de acuerdo con la política del curso.

**Nombre y apellido:** «…»          **Fecha:** «…»